In [606]:
import pandas as pd
import numpy as np
import openpyxl
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm
import researchpy
import pingouin
import seaborn

import matplotlib.pyplot as plt

In [607]:
file = "Project-Data.xlsx"

dfs = {"bc":[], "db": [], "wq_class": [], "ch": [], "asn": [], "wq_reg": []}

for df in dfs:
    for i in range(1,6):
        if df in ["bc", "db", "wq_class"]:
            dfs[df].append(pd.read_excel(file, sheet_name=f'{df}_{i}', names=["Task", "Dataset", "Seed", "Method", "Model", "Level", "Mode", "Accuracy"]))
        else:
            dfs[df].append(pd.read_excel(file, sheet_name=f'{df}_{i}', names=["Task", "Dataset", "Seed", "Method", "Model", "Level", "Mode", "MAE", "R2", "MSE"]))


In [611]:
classification_keys = ["bc", "db", "wq_class"]
regression_keys = ["ch", "asn", "wq_reg"]

classification_df = pd.concat(
    [pd.concat(dfs[key], ignore_index=True) for key in classification_keys],
    ignore_index=True
)

regression_df = pd.concat(
    [pd.concat(dfs[key], ignore_index=True) for key in regression_keys],
    ignore_index=True
)

classification_df


,Task,Dataset,Seed,Method,Model,Level,Mode,Accuracy
0,classification,bc,42,base,logreg,NaN,base,0.973684
1,classification,bc,42,rounding,logreg,-2.0,both,0.938596
2,classification,bc,42,rounding,logreg,-2.0,training_data_only,0.938596
3,classification,bc,42,rounding,logreg,-2.0,testing_data_only,0.622807
4,classification,bc,42,rounding,logreg,-1.0,both,0.964912
...,...,...,...,...,...,...,...,...
5695,classification,wq_class,100,sig figs,rfc,4.0,training_data_only,0.740625
5696,classification,wq_class,100,sig figs,rfc,4.0,testing_data_only,0.709375
5697,classification,wq_class,100,sig figs,rfc,5.0,both,0.718750
5698,classification,wq_class,100,sig figs,rfc,5.0,training_data_only,0.734375


In [ ]:
anova_table_pin_class = pingouin.anova(data=classification_df, dv="Accuracy", between=["Dataset", "Method", "Model", "Mode"], ss_type=2, detailed=True, effsize="np2")
anova_table_pin_class

c:\Users\abelg\anaconda3\envs\ress\Lib\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 60, but rank is 1
  warnings.warn('covariance of constraints does not have full '


,Source,SS,DF,MS,F,p_unc,np2
0,Method,-1.366905e-09,5.0,-2.733809e-10,-8.445989e-09,1.000000,-7.514225e-12
1,Model,1.511831e-08,4.0,3.779578e-09,1.167685e-07,1.000000,8.310925e-11
2,Mode,-4.328844e-09,3.0,-1.442948e-09,-4.457928e-08,1.000000,-2.379677e-11
3,Method * Model,-7.573087e-08,20.0,-3.786543e-09,-1.169837e-07,1.000000,-4.163120e-10
4,Method * Mode,-5.910148e-08,15.0,-3.940099e-09,-1.217277e-07,1.000000,-3.248960e-10
5,Model * Mode,3.214354e-08,12.0,2.678629e-09,8.275511e-08,1.000000,1.767013e-10
6,Method * Model * Mode,3.487150e-01,60.0,5.811917e-03,1.795567e-01,0.671771,1.913308e-03
7,Residual,1.819089e+02,5620.0,3.236814e-02,NaN,NaN,NaN


In [610]:
# pd.concat(dfs["db"], ignore_index=True)
pd.set_option("display.max_rows", 20)

thing = pd.concat(dfs["ch"], ignore_index=True)
thing.isna().sum()

Task        0
Dataset     0
Seed        0
Method      0
Model       0
Level      20
Mode        0
MAE         0
R2          0
MSE         0
dtype: int64

In [ ]:
dataset_key = "ch"
metric = "MAE"

anova_df = pd.concat(dfs[dataset_key], ignore_index=True)

anova_df = anova_df.dropna(subset=[metric])
base_df = anova_df[anova_df["Method"] == "base"]

anova_df = anova_df[anova_df["Method"] != "base"].copy()

anova_df = anova_df.reset_index(drop=True)


In [ ]:
anova_df.dtypes

In [ ]:
anova_table_pin = pingouin.anova(data=anova_df, dv="MAE", between=["Method", "Model", "Mode"], ss_type=2, detailed=True, effsize="np2")
anova_table_pin

In [ ]:
anova_table_pin2 = pingouin.anova(data=anova_df, dv="MAE", between=["Method", "Model", "Mode"], ss_type=3, detailed=True, effsize="np2")
anova_table_pin2

In [ ]:
ms_error = anova_table_pin.loc[anova_table_pin["Source"] == "Residual", "MS"].iloc[0]
ss_error = anova_table_pin.loc[anova_table_pin["Source"] == "Residual", "SS"].iloc[0]

anova_table_pin["partial_omega_sq"] = (
    (anova_table_pin["SS"] - anova_table_pin["DF"] * ms_error) /
    (anova_table_pin["SS"] + ss_error + ms_error)
)

anova_table_pin.loc[anova_table_pin["Source"] == "Residual", "partial_omega_sq"] = None
anova_table_pin["partial_omega_sq"] = anova_table_pin["partial_omega_sq"].clip(lower=0)
anova_table_pin

In [ ]:
counts = (
    anova_df
    .groupby(["Method", "Model", "Mode"])
    .size()
    .reset_index(name="count")
)

counts["count"].nunique()

In [ ]:
respyanova = researchpy.anova(formula_like=f"{metric} ~ C(Method)*C(Model)*C(Mode)", data=anova_df, sum_of_squares=3)
anova_table2 = respyanova.results(return_type="Dataframe", decimals=5)[1]
anova_table2 = anova_table2.replace("", np.nan)
anova_table2


In [ ]:
respyanova2 = researchpy.anova(formula_like=f"{metric} ~ C(Method)*C(Model)*C(Mode)", data=anova_df, sum_of_squares=2)
anova_table5 = respyanova.results(return_type="Dataframe", decimals=5)[1]
anova_table5



In [ ]:
anova_table_pin

In [ ]:
basic_stats = anova_df.groupby("Method")["MAE"].agg(["count", "mean", "std", "min", "max"])
basic_stats

In [ ]:
base_stats = pd.DataFrame(base_df.groupby("Model")["MAE"].agg(["count", "mean", "std", "min", "max"]))
# base_stats.index = ["base_overall"]
base_stats

In [ ]:
wb = openpyxl.load_workbook("Project-Data.xlsx")
ws = wb["ch_ANALYSIS"]

startrow = ws.max_row + 2  # 2 rows after last filled row

with pd.ExcelWriter("Project-Data.xlsx", engine="openpyxl", mode="a", if_sheet_exists="overlay") as writer:
    base_stats.to_excel(writer, sheet_name="ch_ANALYSIS", startrow=startrow)


In [ ]:
sub = anova_df[anova_df["Method"] == "sig figs"].copy()

for col in ["Level", "Model", "Mode"]:
    sub[col] = sub[col].astype("category")

aov = pingouin.anova(data=sub, dv="MAE", between=["Level", "Model", "Mode"], ss_type=2, detailed=True, effsize="np2")
    
ms_error = aov.loc[aov["Source"] == "Residual", "MS"].iloc[0]
ss_error = aov.loc[aov["Source"] == "Residual", "SS"].iloc[0]

aov["partial_omega_sq"] = (
    (aov["SS"] - aov["DF"] * ms_error) /
    (aov["SS"] + ss_error + ms_error)
)

aov.loc[aov["Source"] == "Residual", "partial_omega_sq"] = None
aov["partial_omega_sq"] = aov["partial_omega_sq"].clip(lower=0)

sub.groupby("Level")["MAE"].agg(["count", "mean", "std", "min", "max"])



In [ ]:
poop = None
for method in anova_df["Method"].unique():
    sub = anova_df[anova_df["Method"] == method].copy()

    for col in ["Level", "Model", "Mode"]:
        sub[col] = sub[col].astype("category")
    # print(sub)

    aov = pingouin.anova(data=sub, dv="MAE", between=["Level", "Model", "Mode"], ss_type=2, detailed=True, effsize="np2")
    
    ms_error = aov.loc[aov["Source"] == "Residual", "MS"].iloc[0]
    ss_error = aov.loc[aov["Source"] == "Residual", "SS"].iloc[0]

    aov["partial_omega_sq"] = (
        (aov["SS"] - aov["DF"] * ms_error) /
        (aov["SS"] + ss_error + ms_error)
    )

    aov.loc[aov["Source"] == "Residual", "partial_omega_sq"] = None
    aov["partial_omega_sq"] = aov["partial_omega_sq"].clip(lower=0)

    print(aov)

    # wb = openpyxl.load_workbook("Project-Data.xlsx")
    # ws = wb["ch_ANALYSIS"]

    # startrow = ws.max_row + 2  # 2 rows after last filled row

    # with pd.ExcelWriter("Project-Data.xlsx", engine="openpyxl", mode="a", if_sheet_exists="overlay") as writer:
    #     aov.to_excel(writer, sheet_name="ch_ANALYSIS", startrow=startrow)
    #     print('yy')




In [ ]:
anova_df

In [ ]:
col = "MSE"

# take structure from first df
new_df = dfs["wq_reg"][0].copy()

# replace accuracy with averaged version
new_df[col] = pd.concat(
    [df[col].reset_index(drop=True) for df in dfs["wq_reg"]],
    axis=1
).mean(axis=1)
new_df

In [ ]:

new_df.loc[new_df["MSE"] == new_df["MSE"].max(), "MSE"] = np.nan
new_df.isna().sum()

In [ ]:
metric = "MAMSE" 
dfff = new_df  # change to 'accuracy', 'r2', 'mse', or 'mae'
# seed = Average

datasets = dfff['Dataset'].dropna().unique()
models = dfff['Model'].dropna().unique()
methods = dfff['Method'].dropna().unique()

for dataset in datasets:
    

    # filter once for this dataset
    df_d = dfff[
        (dfff['Dataset'] == dataset) &
        (dfff[metric].notna())
    ]

    if df_d.empty:
        continue

    # decide y-limits based on metric
    if metric == 'Accuracy':
        ymin, ymax = 0, 1

    elif metric == 'r2':
        data_min = df_d[metric].min()
        data_max = df_d[metric].max()
        print(data_min)

        y_range = data_max - data_min
        pad = 0.1 * y_range if y_range > 0 else 0.1

        ymin = data_min - pad
        print(ymin)
        # ymax = data_max + pad

        # optional: cap the top at 1 since R2 usually doesn't exceed 1
        ymax = 1

    elif metric in ['mse', 'mae']:
        data_min = df_d[metric].min()
        data_max = df_d[metric].max()
        print(data_min,data_max)

        y_range = data_max - data_min
        pad = 0.1 * y_range if y_range > 0 else 0.1

        ymin = data_min - pad
        ymax = data_max + pad

    else:
        # fallback for any other metric
        data_min = df_d[metric].min()
        data_max = df_d[metric].max()

        y_range = data_max - data_min
        pad = 0.1 * y_range if y_range > 0 else 0.1

        ymin = data_min - pad
        ymax = data_max + pad

    fig, axes = plt.subplots(
        len(models),
        len(methods),
        figsize=(20, 12),
        sharey=True
    )

    axes = np.atleast_2d(axes)

    for i, model in enumerate(models):
        for j, method in enumerate(methods):

            ax = axes[i, j]

            subset = df_d[
                (df_d['Model'] == model) &
                (df_d['Method'] == method)
            ]

            if subset.empty:
                ax.set_visible(False)
                continue

            for mode in subset['Mode'].dropna().unique():
                mode_data = subset[subset['Mode'] == mode].sort_values('Level')

                ax.plot(
                    mode_data['Level'],
                    mode_data[metric],
                    marker='o',
                    label=mode
                )

            if i == 0:
                ax.set_title(method)
            if j == 0:
                ax.set_ylabel(model)

            ax.set_xlabel("Level")
            ax.set_xticks(sorted(subset['Level'].dropna().unique()))
            ax.set_ylim(ymin, ymax)

            if metric == 'r2':
                ax.axhline(0, linestyle='--', linewidth=1)

            ax.legend()

    fig.suptitle(
        f"Dataset: {dataset} | Average | Metric: {metric}",
        fontsize=16
    )

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

In [ ]:
avg_df = anova_df.groupby("Model")["Accuracy"].mean().reset_index()
avg_df

In [ ]:
plt.style.use("seaborn-v0_8-whitegrid")

plt.rcParams.update({
    "font.size": 12,
    "axes.titlesize": 16,
    "axes.labelsize": 13,
    "legend.fontsize": 11
})

In [ ]:

def plot_single_var(
    df,
    var,
    metric="Accuracy",
    dataset_key=None,
    dataset_label=None,   # 🔥 new
    drop_base=True,
    save_path=None
):
    plot_df = df.copy()

    if dataset_key is not None:
        plot_df = plot_df[plot_df["Dataset"] == dataset_key]

    plot_df = plot_df.dropna(subset=[metric])

    if drop_base and "Method" in plot_df.columns:
        plot_df = plot_df[plot_df["Method"] != "base"]

    # average across everything except var
    avg_df = plot_df.groupby(var)[metric].mean().reset_index()
    avg_df = avg_df.sort_values(metric, ascending=False)

    plt.figure(figsize=(8, 5))
    bars = plt.bar(avg_df[var].astype(str), avg_df[metric])

    # value labels
    for bar in bars:
        y = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2, y, f"{y:.3f}",
                 ha="center", va="bottom")

    plt.xlabel(var, labelpad=10)
    plt.ylabel(metric, labelpad=10)

    # 🔥 title handling
    if dataset_label:
        title = f"{metric} by {var}: {dataset_label}"
    elif dataset_key:
        title = f"{metric} by {var}: {dataset_key}"
    else:
        title = f"{metric} by {var}"

    plt.title(title, pad=10)

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")

    plt.show()

In [ ]:
# Compare models overall
plot_single_var(anova_df, var="Model", dataset_key="bc")

# Compare methods
plot_single_var(anova_df, var="Method", dataset_key="bc")

# Model vs Mode
plot_single_var(anova_df, var="Mode", dataset_key="bc")

In [ ]:
def plot_clean_heatmap(df, dataset_key="bc", metric="Accuracy"):
    plot_df = df.copy()
    plot_df = plot_df[plot_df["Dataset"] == dataset_key]
    plot_df = plot_df.dropna(subset=[metric])
    plot_df = plot_df[plot_df["Method"] != "base"]

    heat = plot_df.groupby(["Mode", "Model"])[metric].mean().unstack()

    fig, ax = plt.subplots(figsize=(7, 4))
    im = ax.imshow(heat.values)

    ax.set_xticks(range(len(heat.columns)))
    ax.set_xticklabels(heat.columns)
    ax.set_yticks(range(len(heat.index)))
    ax.set_yticklabels(heat.index)

    # 🔥 numbers inside
    for i in range(heat.shape[0]):
        for j in range(heat.shape[1]):
            ax.text(j, i, f"{heat.iloc[i,j]:.2f}",
                    ha="center", va="center", color="white")

    fig.colorbar(im, ax=ax)
    plt.title("Model Performance Across Modes")
    plt.tight_layout()
    plt.show()
plot_clean_heatmap(anova_df, dataset_key="bc", metric="Accuracy")

In [ ]:
def plot_clean_lines(df, dataset_key="bc", metric="Accuracy"):
    plot_df = df.copy()
    plot_df = plot_df[plot_df["Dataset"] == dataset_key]
    plot_df = plot_df.dropna(subset=[metric])
    plot_df = plot_df[plot_df["Method"] != "base"]

    avg_df = plot_df.groupby(["Method", "Level"])[metric].mean().reset_index()

    plt.figure(figsize=(8, 5))

    for method in avg_df["Method"].unique():
        sub = avg_df[avg_df["Method"] == method].sort_values("Level")
        plt.plot(
            sub["Level"], sub[metric],
            marker="o",
            linewidth=2,
            label=method
        )

    plt.title("Precision Method Trends")
    plt.xlabel("Level")
    plt.ylabel(metric)
    plt.legend(frameon=False)
    plt.tight_layout()
    plt.show()
plot_clean_lines(anova_df, dataset_key="bc", metric="Accuracy")